In [ ]:
import os
import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

from pathlib import Path

SAVE = False  # whether to save the outputs of this notebook

if SAVE:
    # Replace this with the path to directory where you would like results to be saved
    OUTPUT_PATH = "../output_files/synthetic_example"
    os.makedirs(OUTPUT_PATH, exist_ok=True)  # Create path if it doesn't exist


# data paths 
DATA = Path('data')

In [ ]:
import pandas as pd
from pathlib import Path

# creating experiment 

files = {
    'lsm1_c1': DATA / 'lsm1_clone1.txt',
    'lsm1_c2': DATA / 'lsm1_clone2.txt',
    'lsm1_down1': DATA / 'down_lsm1_R.txt',
    'lsm1_down2': DATA / 'bgr_LSM1_R.txt', # Przykład: dodajemy kolejną próbkę lsm1
    'ski2_down': DATA / 'down_ski2_R.txt',
    'ski2_bgr': DATA / 'bgr_SKI2_R.txt'
}

# load counts
all_counts = {}
for sample_name, file_path in files.items():
    df = pd.read_csv(file_path, sep=r'\s+', header=None, names=['gene_id', sample_name])
    df['gene_id'] = df['gene_id'].astype(str).str.strip()
    all_counts[sample_name] = df.set_index('gene_id')[sample_name]

# we switch type for experiments vs genes 
counts_df = pd.DataFrame(all_counts).T.fillna(0).astype(int)

# create condition map
condition_map = {
    'lsm1_c1': 'control', 'lsm1_c2': 'control',
    'lsm1_down1': 'treatment', 'lsm1_down2': 'treatment',
    'ski2_down': 'ski_exp', 'ski2_bgr': 'ski_exp'
}

# creating df with experiment metadata
metadata = pd.DataFrame({
    'condition': [condition_map[s] for s in counts_df.index]
}, index=counts_df.index)

print("\nMetadata:")
print(metadata)

In [ ]:
# --- FILTERING ---
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 1]
counts_df = counts_df[genes_to_keep]


print(f"Remain genes: {counts_df.shape[1]}")

In [ ]:
# *** baseline 
# preparation of dataset
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="condition",
    refit_cooks=True,
    inference=inference,
    # n_cpus=8, # n_cpus can be specified here or in the inference object
)

In [ ]:
# *** baseline 
# preparation of dataset
inference = DefaultInference(n_cpus=8)
dds_diff = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="condition",
    refit_cooks=True,
    inference=inference,
    # n_cpus=8, # n_cpus can be specified here or in the inference object
)

In [ ]:
dds_diff.deseq2()

In [ ]:
dds.deseq2()

In [ ]:
print(dds)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- STEP 2: Extract Results (Contrast: Treatment vs Control) ---
# Logic: We define which groups to compare. "control" is our reference.
# Technicznie: contrast=['kolumna', 'grupa_badana', 'referencja']
# TODO dla dwóch różnych trzeba
stat_res = DeseqStats(dds_diff, contrast=["condition", "treatment", "control"], inference=inference)
stat_res.summary()
res_df = stat_res.results_df

# Clean up: Remove non-gene entries (alignment metrics)
res_df = res_df[~res_df.index.str.startswith('__')]

# --- STEP 3: Filtering UP and DOWN regulated genes ---
# Logic: We use p-adjusted < 0.05 and absolute log2FoldChange > 1 (2-fold change).
# Tłumaczenie: Szukamy genów, gdzie zmiana jest istotna statystycznie i co najmniej dwukrotna.

up_regulated = res_df[(res_df.padj < 0.05) & (res_df.log2FoldChange > 1)]
down_regulated = res_df[(res_df.padj < 0.05) & (res_df.log2FoldChange < -1)]

# Save to files for functional analysis (e.g., g:Profiler)
# Pliki tekstowe zawierające tylko nazwy genów (identyfikatory)
with open("up_lsm1.txt", "w") as f:
    f.write("\n".join(up_regulated.index))

with open("down_lsm1.txt", "w") as f:
    f.write("\n".join(down_regulated.index))

with open("bgr_lsm1.txt", "w") as f:
    f.write("\n".join(res_df.index))

print(f"Analysis complete. UP: {len(up_regulated)}, DOWN: {len(down_regulated)}")

# --- STEP 4: Visualization (Volcano Plot) ---
# Logic: X-axis shows the magnitude of change, Y-axis shows statistical significance.
# Problem techniczny: padj może wynosić 0, co psuje log10. Używamy małego offsetu.

plt.figure(figsize=(8, 6))
plt.scatter(res_df.log2FoldChange, -np.log10(res_df.padj + 1e-20), c='grey', alpha=0.5, s=10, label='Non-sig')
plt.scatter(up_regulated.log2FoldChange, -np.log10(up_regulated.padj + 1e-20), c='red', s=10, label='UP')
plt.scatter(down_regulated.log2FoldChange, -np.log10(down_regulated.padj + 1e-20), c='blue', s=10, label='DOWN')

plt.axhline(-np.log10(0.05), color='black', linestyle='--', alpha=0.5)
plt.axvline(1, color='black', linestyle='--', alpha=0.5)
plt.axvline(-1, color='black', linestyle='--', alpha=0.5)

plt.title("Volcano Plot: LSM1 Treatment vs Control")
plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted P-value")
plt.legend()
plt.show()

Co zrobić, przeprowadzenie analizy różnicowej, otrzymać plik z analizy deseq, przyjąć wartości graniczne (p value albo poprawka) oraz wybrać taka grupę genów, które mają:
- obniżoną ekspresje
- podwyższoną ekspresje 

Potem przeprowadzic analize funkcjonalną
- (przeglądarka - gprofiler) ale nie tego
- z pythona - jak najwygodniej 


#TODO 
- porównać robienie tego w R i w Pythonie

S batchem a nie s runem ??? 
- bardzo cwanie można sobie wybrać procesory podłączone do karty graficznej (bo mamy )

PLAN
- zrobimy te rzeczy i potem wyodrębnienie genów takich, że:
  - które są down albo up regulowane i zapisać to do pliku
  - jak to będziemy mieli to przejdziemy do analizy funkcjonalnej 

In [ ]:
list(up_regulated.index)